# SQL Views for Power BI

Views built directly on top of the loaded tables, so Power BI queries clean,
pre-joined data rather than raw tables. Five views: a statewide overview,
a Reno-vs-state indexed valuation series (solves the "flagship chart" gap
without needing raw Reno dollar figures), county-level levy change, latest
county snapshot, and an indexed neighbor-state comparison.

In [3]:
import sqlite3
import pandas as pd


conn = sqlite3.connect("../data/processed/kansas_tax.db")
cur = conn.cursor()

views_sql = """
DROP VIEW IF EXISTS vw_statewide_overview;
CREATE VIEW vw_statewide_overview AS
SELECT 
    a.year,
    a.total_appraised_billions,
    v.total_assessed_value_billions,
    t.total_tax_millions,
    ROUND(v.total_assessed_value_billions / a.total_appraised_billions * 100, 2) AS assessed_pct_of_appraised,
    t.residential_pct AS residential_pct_of_tax
FROM statewide_appraised a
JOIN statewide_valuation v ON a.year = v.year
JOIN statewide_tax t ON a.year = t.year
ORDER BY a.year;

DROP VIEW IF EXISTS vw_reno_valuation_index;
CREATE VIEW vw_reno_valuation_index AS
WITH RECURSIVE indexed AS (
    SELECT county, year, valuation_change_pct, 100.0 AS index_value
    FROM reno_valuation_vs_state
    WHERE year = (SELECT MIN(year) FROM reno_valuation_vs_state)

    UNION ALL

    SELECT r.county, r.year, r.valuation_change_pct,
           i.index_value * (1 + r.valuation_change_pct / 100.0)
    FROM reno_valuation_vs_state r
    JOIN indexed i ON r.county = i.county AND r.year = i.year + 1
)
SELECT * FROM indexed ORDER BY county, year;

DROP VIEW IF EXISTS vw_county_levy_latest;
CREATE VIEW vw_county_levy_latest AS
SELECT county, year, rural_levy, urban_levy, county_avg_levy
FROM mill_levies
WHERE year = (SELECT MAX(year) FROM mill_levies)
ORDER BY county_avg_levy DESC;

DROP VIEW IF EXISTS vw_county_levy_change;
CREATE VIEW vw_county_levy_change AS
WITH first_year AS (
    SELECT county, county_avg_levy AS levy_start
    FROM mill_levies WHERE year = (SELECT MIN(year) FROM mill_levies)
),
last_year AS (
    SELECT county, county_avg_levy AS levy_latest
    FROM mill_levies WHERE year = (SELECT MAX(year) FROM mill_levies)
)
SELECT f.county, f.levy_start, l.levy_latest,
       ROUND((l.levy_latest - f.levy_start) / f.levy_start * 100, 2) AS pct_change
FROM first_year f
JOIN last_year l ON f.county = l.county
ORDER BY pct_change DESC;

DROP VIEW IF EXISTS vw_neighbor_state_index;
CREATE VIEW vw_neighbor_state_index AS
WITH base AS (
    SELECT state, tax_bill AS base_bill
    FROM neighbor_state_comparison
    WHERE year = 2023
)
SELECT n.state, n.year, n.tax_bill, n.median_home_value,
       ROUND(n.tax_bill / b.base_bill * 100, 2) AS tax_bill_index
FROM neighbor_state_comparison n
JOIN base b ON n.state = b.state
ORDER BY n.state, n.year;
"""

cur.executescript(views_sql)
conn.commit()
print("Views created.")

Views created.


In [4]:
for view in ["vw_statewide_overview", "vw_reno_valuation_index", "vw_county_levy_latest", "vw_county_levy_change", "vw_neighbor_state_index"]:
    result = pd.read_sql(f"SELECT * FROM {view} LIMIT 3", conn)
    print(f"\n{view}:")
    print(result)

conn.close()


vw_statewide_overview:
   year  total_appraised_billions  total_assessed_value_billions  \
0  1998                   106.790                         18.849   
1  1999                   112.926                         19.608   
2  2000                   121.886                         20.875   

   total_tax_millions  assessed_pct_of_appraised  residential_pct_of_tax  
0            1964.549                      17.65                   40.59  
1            2105.586                      17.36                   41.63  
2            2303.781                      17.13                   42.53  

vw_reno_valuation_index:
  county  year  valuation_change_pct  index_value
0   Reno  2013                   2.2      100.000
1   Reno  2014                   2.4      102.400
2   Reno  2015                   1.5      103.936

vw_county_levy_latest:
     county  year  rural_levy  urban_levy  county_avg_levy
0   Greeley  2025     259.498     311.480          270.739
1   Stanton  2025     239.699     2